# Stage 2 — The score spine

**Checkpoint:** `stage-2`

One score, reused by every decision app you build after this. It ships only if it clears a
**hard gate: held-out AUC ≥ 0.78.** This notebook shows what the score looks like, whether it
rank-orders, and how it explains itself.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- stage guard: fail loudly and usefully, not mysteriously ---------------
NEEDS = [
    "score/models/scorecard.pkl",
    "score/models/metadata.json",
    "score/models/score_scaling.json",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-2 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + (Path("stage.txt").read_text().strip())
        + ". Fix with either:\n"
        "  git checkout stage-2      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-2 artifacts present.")

## 1. The gate — did it pass, and by how much?

In [ ]:
import json
meta = json.loads(Path("score/models/metadata.json").read_text())

auc = float(meta["metrics"]["auc"])
ks  = float(meta["metrics"]["ks"])
AUC_GATE = float(meta["gate"]["auc_min"])   # the gate committed alongside the model

print(f"held-out AUC : {auc:.4f}")
print(f"KS           : {ks:.4f}   (reported, not gated)")
print(f"gate         : AUC >= {AUC_GATE}")
print(f"result       : {'PASS  — margin +' + format(auc - AUC_GATE, '.4f') if auc >= AUC_GATE else 'FAIL'}")

fig, ax = plt.subplots(figsize=(7, 1.7))
ax.barh([0], [auc], color="#2f7d4f" if auc >= AUC_GATE else "#b3372e")
ax.axvline(AUC_GATE, color="#b3372e", ls="--", lw=2, label=f"gate {AUC_GATE}")
ax.set_xlim(.5, 1.0); ax.set_yticks([]); ax.set_xlabel("held-out AUC"); ax.legend(loc="lower right")
ax.set_title("The gate is a committed promise, not a preference")
plt.tight_layout(); plt.show()

> **A gate you can move is not a gate.** If yours fails, investigate the model — never
> negotiate the threshold. The gate that counts is the one committed in the tag.

## 2. Score the whole population with the saved model

In [ ]:
from shared.config import RAW
from score.src.predict import predict_score_pd, _BANDS_BINS, _BANDS_LABELS

businesses = pd.read_parquet(RAW / "businesses.parquet")
scored = predict_score_pd(businesses)          # business_score, pd, score_band
df = pd.concat([businesses[["business_id", "industry", "default"]].reset_index(drop=True),
                scored.reset_index(drop=True)], axis=1)
df.head()

## 3. The score distribution, banded

In [ ]:
BAND_COLORS = {"D": "#b3372e", "C": "#c9782a", "B": "#c9a227", "A": "#4f9e56", "AAA": "#2f7d4f"}

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.hist(df["business_score"], bins=54, color="#9fb4d4", edgecolor="white", linewidth=.6)
for lo, hi, lab in zip(_BANDS_BINS[:-1], _BANDS_BINS[1:], _BANDS_LABELS):
    ax.axvspan(lo, hi, color=BAND_COLORS[lab], alpha=.10)
    ax.text((lo + hi) / 2, ax.get_ylim()[1] * .94, lab, ha="center",
            fontsize=13, weight="bold", color=BAND_COLORS[lab])
ax.set_xlim(_BANDS_BINS[0], _BANDS_BINS[-1])
ax.set_xlabel("business credit score (300–850)"); ax.set_ylabel("borrowers")
ax.set_title("Every borrower gets a score")
plt.tight_layout(); plt.show()

## 4. Does it rank-order? — the table a validator reads before any AUC

In [ ]:
by_band = (df.groupby("score_band", observed=True)
             .agg(borrowers=("default", "size"), default_rate=("default", "mean"))
             .reindex(_BANDS_LABELS).dropna())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(by_band.index, by_band["default_rate"] * 100,
       color=[BAND_COLORS[b] for b in by_band.index])
for i, v in enumerate(by_band["default_rate"]):
    ax.text(i, v * 100 + .4, f"{v:.1%}", ha="center", weight="bold")
ax.set_ylabel("default rate (%)"); ax.set_title("Riskier band → higher default. Monotonic = the score works.")
plt.tight_layout(); plt.show()

monotonic = by_band["default_rate"].is_monotonic_decreasing
print(("OK — monotonic across all bands." if monotonic else
       "WARNING — not monotonic; a validator will ask why."))
by_band.style.format({"default_rate": "{:.2%}", "borrowers": "{:,.0f}"})

## 5. ROC and KS — the same model, two views

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y, p = df["default"].to_numpy(), df["pd"].to_numpy()
fpr, tpr, _ = roc_curve(y, p)
pop_auc = roc_auc_score(y, p)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(fpr, tpr, color="#245bb2", lw=2, label=f"AUC = {pop_auc:.4f}")
axes[0].plot([0, 1], [0, 1], ls="--", color="#999", lw=1)
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC (whole population)"); axes[0].legend()

order = np.argsort(p)
cum_bad  = np.cumsum(y[order]) / y.sum()
cum_good = np.cumsum(1 - y[order]) / (len(y) - y.sum())
xs = np.arange(len(y)) / len(y)
axes[1].plot(xs, cum_bad, label="cumulative bads", color="#b3372e")
axes[1].plot(xs, cum_good, label="cumulative goods", color="#2f7d4f")
k = int(np.argmax(np.abs(cum_bad - cum_good)))
axes[1].vlines(xs[k], cum_good[k], cum_bad[k], color="#245bb2", lw=2,
               label=f"KS = {abs(cum_bad[k] - cum_good[k]):.4f}")
axes[1].set_xlabel("population sorted by score"); axes[1].set_title("KS separation"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"NOTE: population AUC {pop_auc:.4f} is computed on ALL rows including those the model")
print(f"      trained on. The honest number is the held-out {auc:.4f} above.")

## 6. The score explains itself

For a linear-in-WoE model each feature's push on the odds is **exactly** `WoE(bin) × coefficient`
— arithmetic, not an approximation of it. That is what makes an adverse-action letter defensible.

In [ ]:
import joblib
from score.src.feature_engineering import FEATURE_COLUMNS, compute_features
from score.src.reason_codes import feature_contributions

scorecard = joblib.load("score/models/scorecard.pkl")
X = compute_features(businesses)[FEATURE_COLUMNS]

i = int(df["business_score"].idxmin())          # a genuinely risky applicant
contrib = feature_contributions(scorecard, X.iloc[[i]]).iloc[0].sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ["#b3372e" if v > 0 else "#2f7d4f" for v in contrib.values]
ax.barh(contrib.index, contrib.values, color=colors)
ax.axvline(0, color="#333", lw=1)
ax.set_xlabel("push on the odds of default  (→ riskier)")
ax.set_title(f"Why {df.loc[i, 'business_id']} scored {df.loc[i, 'business_score']}  "
             f"(band {df.loc[i, 'score_band']})")
plt.tight_layout(); plt.show()

print("Top 3 adverse reasons (this is the adverse-action letter):")
for feat, val in contrib.sort_values(ascending=False).head(3).items():
    print(f"  {feat:<28} impact {val:+.4f}")

---
**Next:** `03_stage3_decisions_and_price.ipynb` — turn this score into an approve/decline and
a price, and find the money.